# Stage 2 — HAM10000: Lesion-Aware Splitting and a 10-Backbone Transfer Learning Benchmark

**Why this dataset.** Stage 1b established that the hair/scalp dataset holds ~250 unique
photographs behind 10,800 files. HAM10000 ships a `lesion_id` column, so grouping is
**ground truth from metadata** — no similarity threshold to calibrate or defend.

**What this notebook produces**
1. A verified lesion-level 70/15/15 split with zero lesion overlap (asserted, not assumed)
2. A naive image-level split for the leakage ablation
3. Ten ImageNet-pretrained backbones fine-tuned under an identical protocol
4. A results table with accuracy, **macro-F1**, balanced accuracy and per-class recall

**Read this before running.** HAM10000 is severely imbalanced — `nv` is ~67% of images.
A model predicting `nv` for everything scores 67% accuracy and is worthless. **Macro-F1 is
the headline metric here, not accuracy.** Report both; lead with macro-F1.

**Runtime** ~4–7 h for all ten on a Kaggle T4x2. Results are checkpointed after every model,
and re-running skips completed ones, so a session timeout costs you one model, not the run.

In [1]:
import os, json, gc, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

DATA   = Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")
OUT    = Path("/kaggle/working/stage2"); OUT.mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(exist_ok=True); (OUT / "models").mkdir(exist_ok=True)

SEED        = 42
IMG_SIZE    = 224
BATCH       = 32
EPOCHS_HEAD = 8      # frozen backbone
EPOCHS_FT   = 12     # fine-tuning
FT_FRACTION = 0.30   # top fraction of backbone layers unfrozen
LR_HEAD     = 1e-3
LR_FT       = 1e-5
MIXED_PREC  = True   # big speedup on T4; harmless on P100
RUN_ABLATION_MODELS = ["DenseNet121", "EfficientNetB0", "ResNet50"]

keras.utils.set_random_seed(SEED)
if MIXED_PREC:
    keras.mixed_precision.set_global_policy("mixed_float16")
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)


TF 2.20.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 1. Metadata and lesion-level split

`lesion_id` is the grouping key: one physical lesion often contributes several images.
Splitting on images rather than lesions puts different photographs of the *same* lesion on
both sides of the boundary — the exact failure mode Stage 1 was fighting, except here the
metadata hands us the answer.

In [2]:
meta = pd.read_csv(DATA / "HAM10000_metadata.csv")

img_paths = {}
for d in DATA.rglob("*"):
    if d.is_dir() and d.name.startswith("HAM10000_images"):
        for f in d.glob("*.jpg"):
            img_paths[f.stem] = str(f)
if not img_paths:
    for f in DATA.rglob("*.jpg"):
        img_paths[f.stem] = str(f)
meta["path"] = meta["image_id"].map(img_paths)
missing = meta["path"].isna().sum()
assert missing == 0, f"{missing} images not found — check DATA path"

CLASSES = sorted(meta["dx"].unique())
NUM_CLASSES = len(CLASSES)
meta["y"] = meta["dx"].map({c: i for i, c in enumerate(CLASSES)})

print(f"{len(meta)} images | {meta['lesion_id'].nunique()} unique lesions | {NUM_CLASSES} classes")
print(f"images per lesion: mean {len(meta)/meta['lesion_id'].nunique():.2f}, "
      f"max {meta.groupby('lesion_id').size().max()}")
print("\nclass distribution:")
dist = meta["dx"].value_counts()
for c, n in dist.items():
    print(f"  {c:<6} {n:>5}  ({n/len(meta):.1%})")
print(f"\nmajority-class baseline accuracy = {dist.max()/len(meta):.4f}  <-- beat this or it means nothing")

10015 images | 7470 unique lesions | 7 classes
images per lesion: mean 1.34, max 6

class distribution:
  nv      6705  (66.9%)
  mel     1113  (11.1%)
  bkl     1099  (11.0%)
  bcc      514  (5.1%)
  akiec    327  (3.3%)
  vasc     142  (1.4%)
  df       115  (1.1%)

majority-class baseline accuracy = 0.6695  <-- beat this or it means nothing


In [3]:
# Lesion-aware split: stratified by diagnosis, grouped by lesion.
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
trval_idx, test_idx = next(sgkf.split(meta, meta["y"], groups=meta["lesion_id"]))

sub = meta.iloc[trval_idx].reset_index(drop=True)
sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
tr_idx, val_idx = next(sgkf2.split(sub, sub["y"], groups=sub["lesion_id"]))

meta["split"] = None
meta.loc[meta.index[test_idx], "split"] = "test"
meta.loc[meta.index[trval_idx[tr_idx]], "split"] = "train"
meta.loc[meta.index[trval_idx[val_idx]], "split"] = "val"

# INTEGRITY GATE — assert, don't hope.
les = {s: set(meta.loc[meta["split"] == s, "lesion_id"]) for s in ["train", "val", "test"]}
overlap_tt = les["train"] & les["test"]
overlap_tv = les["train"] & les["val"]
overlap_vt = les["val"] & les["test"]
print(f"lesion overlap  train/test={len(overlap_tt)}  train/val={len(overlap_tv)}  val/test={len(overlap_vt)}")
assert not (overlap_tt or overlap_tv or overlap_vt), "LESION LEAKAGE — do not train"

ids = {s: set(meta.loc[meta["split"] == s, "image_id"]) for s in ["train", "val", "test"]}
assert not (ids["train"] & ids["test"]), "image-level overlap"
print("INTEGRITY GATE: PASS — zero lesion overlap across splits\n")
print(meta.groupby(["split", "dx"]).size().unstack(fill_value=0).loc[["train", "val", "test"]])

lesion overlap  train/test=0  train/val=0  val/test=0
INTEGRITY GATE: PASS — zero lesion overlap across splits

dx     akiec  bcc  bkl  df  mel    nv  vasc
split                                      
train    244  358  780  79  814  4769   101
val       49   67  151  27  153   962    20
test      34   89  168   9  146   974    21


### 1b. The naive split, for the ablation

Same proportions, split on images instead of lesions. The lesion-crossing count below is the
quantity that inflates published numbers on this dataset.

In [4]:
n_tr, n_val = (meta["split"] == "train").sum(), (meta["split"] == "val").sum()
naive_tr, naive_tmp = train_test_split(
    meta, train_size=n_tr, stratify=meta["y"], random_state=SEED)
naive_val, naive_te = train_test_split(
    naive_tmp, train_size=n_val, stratify=naive_tmp["y"], random_state=SEED)
meta["naive_split"] = None
meta.loc[naive_tr.index, "naive_split"] = "train"
meta.loc[naive_val.index, "naive_split"] = "val"
meta.loc[naive_te.index, "naive_split"] = "test"

crossing = len(set(naive_tr["lesion_id"]) & set(naive_te["lesion_id"]))
n_les_te = naive_te["lesion_id"].nunique()
print(f"naive split: {crossing}/{n_les_te} test lesions ({crossing/n_les_te:.1%}) "
      f"also appear in train")
meta.to_csv(OUT / "splits.csv", index=False)

naive split: 461/1374 test lesions (33.6%) also appear in train


## 2. Decode every image once into RAM

10,015 images at 224x224 uint8 is ~1.5 GB — it fits, and it removes JPEG decoding from the
training loop entirely. Across twenty-odd training runs this is the difference between a
four-hour benchmark and a fifteen-hour one.

In [5]:
Y     = meta["y"].to_numpy().astype(np.int32)
PATHS = meta["path"].to_numpy()
print(f"streaming {len(PATHS)} images from disk — no RAM cache")

streaming 10015 images from disk — no RAM cache


## 3. Input pipeline

Augmentation is applied **inside the training pipeline only**, after the split — never
written to disk. That is what turned the hair dataset into 43 copies of every photograph.

In [6]:
AUTOTUNE = tf.data.AUTOTUNE

def _decode(path, y):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(img, (IMG_SIZE, IMG_SIZE)), y

def augment(img, y):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, tf.int32))
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)
    return tf.clip_by_value(img, 0.0, 255.0), y

def make_ds(idx, preprocess, training):
    ds = tf.data.Dataset.from_tensor_slices((PATHS[idx], Y[idx]))
    if training:                       # shuffles path strings, not images — a few MB
        ds = ds.shuffle(len(idx), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(_decode, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda i, y: (preprocess(i), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

## 4. The model zoo

Ten backbones spanning the architectural families a reviewer will expect: a plain deep CNN
(VGG16), residual (ResNet50), densely connected (DenseNet121/201), multi-branch
(InceptionV3, InceptionResNetV2), depthwise-separable (Xception), mobile-efficient
(MobileNetV2), compound-scaled (EfficientNetB0, EfficientNetV2B0) and a modern
transformer-influenced convnet (ConvNeXtTiny).

Each family's own `preprocess_input` is used. This matters more than it looks — EfficientNet
and ConvNeXt carry rescaling *inside* the model and expect raw 0–255 input, while ResNet and
VGG expect mean-subtracted BGR. Feeding all ten the same normalisation is the single most
common way these benchmarks get quietly corrupted.

In [7]:
A = keras.applications
ZOO = {
    "VGG16":             (A.VGG16,             A.vgg16.preprocess_input),
    "ResNet50":          (A.ResNet50,          A.resnet.preprocess_input),
    "DenseNet121":       (A.DenseNet121,       A.densenet.preprocess_input),
    "DenseNet201":       (A.DenseNet201,       A.densenet.preprocess_input),
    "InceptionV3":       (A.InceptionV3,       A.inception_v3.preprocess_input),
    "InceptionResNetV2": (A.InceptionResNetV2, A.inception_resnet_v2.preprocess_input),
    "Xception":          (A.Xception,          A.xception.preprocess_input),
    "MobileNetV2":       (A.MobileNetV2,       A.mobilenet_v2.preprocess_input),
    "EfficientNetB0":    (A.EfficientNetB0,    A.efficientnet.preprocess_input),
    "EfficientNetV2B0":  (A.EfficientNetV2B0,  A.efficientnet_v2.preprocess_input),
}
try:
    ZOO["ConvNeXtTiny"] = (A.ConvNeXtTiny, A.convnext.preprocess_input)
except AttributeError:
    print("ConvNeXt unavailable in this TF build — continuing with 10 backbones")

print(f"{len(ZOO)} backbones queued: {', '.join(ZOO)}")

def build(fn, n_classes):
    base = fn(weights="imagenet", include_top=False,
              input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    base.trainable = False
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = base(inp, training=False)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.Dropout(0.3)(x)
    out = keras.layers.Dense(n_classes, activation="softmax", dtype="float32")(x)
    return keras.Model(inp, out), base

11 backbones queued: VGG16, ResNet50, DenseNet121, DenseNet201, InceptionV3, InceptionResNetV2, Xception, MobileNetV2, EfficientNetB0, EfficientNetV2B0, ConvNeXtTiny


## 5. Train / evaluate one backbone

Two phases: train the head on frozen features, then unfreeze the top 30% of the backbone at
a much lower learning rate. **BatchNorm layers stay frozen during fine-tuning** — updating
their statistics on a small, imbalanced dataset is a classic way to destroy a pretrained
backbone, and it shows up as validation loss diverging while training loss falls.

Class weights handle the imbalance; early stopping monitors **val macro-F1**, not val
accuracy, because accuracy is dominated by `nv`.

In [8]:
class MacroF1(keras.callbacks.Callback):
    def __init__(self, ds, y_true):
        super().__init__(); self.ds, self.y = ds, y_true
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.ds, verbose=0).argmax(1)
        logs = logs or {}
        logs["val_macro_f1"] = f1_score(self.y, p, average="macro", zero_division=0)
        print(f"  val_macro_f1 {logs['val_macro_f1']:.4f}")

def run_model(name, split_col="split", tag="lesion"):
    fn, prep = ZOO[name]
    idx = {s: np.where(meta[split_col] == s)[0] for s in ["train", "val", "test"]}
    tr, va, te = (make_ds(idx["train"], prep, True),
                  make_ds(idx["val"], prep, False),
                  make_ds(idx["test"], prep, False))
    yva, yte = Y[idx["val"]], Y[idx["test"]]

    cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=Y[idx["train"]])
    cw = dict(enumerate(cw))

    model, base = build(fn, NUM_CLASSES)
    cbs = lambda: [MacroF1(va, yva),
                   keras.callbacks.EarlyStopping("val_macro_f1", mode="max", patience=4,
                                                 restore_best_weights=True, verbose=1),
                   keras.callbacks.ReduceLROnPlateau("val_macro_f1", mode="max",
                                                     factor=0.3, patience=2, min_lr=1e-7)]

    model.compile(optimizer=keras.optimizers.Adam(LR_HEAD),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    model.fit(tr, validation_data=va, epochs=EPOCHS_HEAD,
              class_weight=cw, callbacks=cbs(), verbose=2)

    base.trainable = True
    cut = int(len(base.layers) * (1 - FT_FRACTION))
    for l in base.layers[:cut]:
        l.trainable = False
    for l in base.layers:                       # keep BN statistics frozen
        if isinstance(l, keras.layers.BatchNormalization):
            l.trainable = False
    n_tr_layers = sum(l.trainable for l in base.layers)
    model.compile(optimizer=keras.optimizers.Adam(LR_FT),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    hist = model.fit(tr, validation_data=va, epochs=EPOCHS_FT,
                     class_weight=cw, callbacks=cbs(), verbose=2)

    prob = model.predict(te, verbose=0)
    pred = prob.argmax(1)
    try:
        auc = roc_auc_score(yte, prob, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")
    res = {
        "model": name, "split": tag,
        "params_M": round(model.count_params() / 1e6, 2),
        "unfrozen_layers": int(n_tr_layers),
        "test_acc": accuracy_score(yte, pred),
        "test_macro_f1": f1_score(yte, pred, average="macro", zero_division=0),
        "test_weighted_f1": f1_score(yte, pred, average="weighted", zero_division=0),
        "test_balanced_acc": balanced_accuracy_score(yte, pred),
        "test_macro_auc": auc,
    }
    for c, r in zip(CLASSES, f1_score(yte, pred, average=None, zero_division=0)):
        res[f"f1_{c}"] = r
    np.save(OUT / f"probs_{tag}_{name}.npy", prob)
    if tag == "lesion":
        model.save(OUT / "models" / f"{name}.keras")
    del model, base; keras.backend.clear_session(); gc.collect()
    return res, pred, yte

## 6. Run the benchmark

Results append to `results.csv` after each model. Re-running the cell skips anything already
recorded, so a timed-out session resumes instead of restarting.

In [9]:
RES = OUT / "results.csv"
done = set()
if RES.exists():
    prev = pd.read_csv(RES)
    done = set(zip(prev["model"], prev["split"]))
    print(f"resuming — {len(done)} runs already complete")

preds = {}
for name in ZOO:
    if (name, "lesion") in done:
        print(f"skip {name}"); continue
    print(f"\n{'='*70}\n{name}  [lesion-aware split]\n{'='*70}")
    t = time.time()
    r, p, yt = run_model(name, "split", "lesion")
    r["minutes"] = round((time.time() - t) / 60, 1)
    preds[name] = (p, yt)
    pd.DataFrame([r]).to_csv(RES, mode="a", header=not RES.exists(), index=False)
    print(f"-> macro-F1 {r['test_macro_f1']:.4f} | acc {r['test_acc']:.4f} "
          f"| bal-acc {r['test_balanced_acc']:.4f} | {r['minutes']} min")


VGG16  [lesion-aware split]


I0000 00:00:1785738364.583367      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785738364.586253      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/8


I0000 00:00:1785738381.500920      71 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  val_macro_f1 0.2950
224/224 - 72s - 323ms/step - accuracy: 0.3292 - loss: 2.7300 - val_accuracy: 0.5038 - val_loss: 1.4091 - val_macro_f1: 0.2950 - learning_rate: 0.0010
Epoch 2/8
  val_macro_f1 0.3378
224/224 - 38s - 170ms/step - accuracy: 0.4267 - loss: 1.6543 - val_accuracy: 0.4997 - val_loss: 1.4502 - val_macro_f1: 0.3378 - learning_rate: 0.0010
Epoch 3/8
  val_macro_f1 0.3430
224/224 - 36s - 162ms/step - accuracy: 0.4684 - loss: 1.4245 - val_accuracy: 0.4255 - val_loss: 1.3924 - val_macro_f1: 0.3430 - learning_rate: 0.0010
Epoch 4/8
  val_macro_f1 0.3420
224/224 - 33s - 145ms/step - accuracy: 0.4738 - loss: 1.3903 - val_accuracy: 0.4962 - val_loss: 1.3515 - val_macro_f1: 0.3420 - learning_rate: 0.0010
Epoch 5/8
  val_macro_f1 0.3732
224/224 - 36s - 160ms/step - accuracy: 0.4778 - loss: 1.3092 - val_accuracy: 0.5451 - val_loss: 1.2399 - val_macro_f1: 0.3732 - learning_rate: 0.0010
Epoch 6/8
  val_macro_f1 0.3464
224/224 - 33s - 149ms/step - accuracy: 0.5031 - loss: 1.3254 - val_a

...

  val_macro_f1 0.4321
224/224 - 87s - 390ms/step - accuracy: 0.4483 - loss: 1.6044 - val_accuracy: 0.6172 - val_loss: 1.0479 - val_macro_f1: 0.4321 - learning_rate: 0.0010
Epoch 2/8
  val_macro_f1 0.4138
224/224 - 31s - 139ms/step - accuracy: 0.5272 - loss: 1.3124 - val_accuracy: 0.5801 - val_loss: 1.0822 - val_macro_f1: 0.4138 - learning_rate: 0.0010
Epoch 3/8
  val_macro_f1 0.4545
224/224 - 27s - 120ms/step - accuracy: 0.5657 - loss: 1.1941 - val_accuracy: 0.6564 - val_loss: 0.9375 - val_macro_f1: 0.4545 - learning_rate: 0.0010
Epoch 4/8
  val_macro_f1 0.4419
224/224 - 26s - 116ms/step - accuracy: 0.5570 - loss: 1.2022 - val_accuracy: 0.5878 - val_loss: 1.1015 - val_macro_f1: 0.4419 - learning_rate: 0.0010
Epoch 5/8
  val_macro_f1 0.4671
224/224 - 26s - 115ms/step - accuracy: 0.5926 - loss: 1.0758 - val_accuracy: 0.6599 - val_loss: 0.9014 - val_macro_f1: 0.4671 - learning_rate: 0.0010
Epoch 6/8
  val_macro_f1 0.4826
224/224 - 41s - 185ms/step - accuracy: 0.5964 - loss: 1.0282 - val_a

2026-08-03 09:03:56.763306: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-03 09:03:56.899214: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-03 09:04:37.055184: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-03 09:04:37.190330: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-03 09:04:52.790468: E external/local_xla/xla/stream_

  val_macro_f1 0.3734
224/224 - 85s - 381ms/step - accuracy: 0.4160 - loss: 1.8072 - val_accuracy: 0.4927 - val_loss: 1.3416 - val_macro_f1: 0.3734 - learning_rate: 0.0010
Epoch 2/8
  val_macro_f1 0.3784
224/224 - 46s - 207ms/step - accuracy: 0.4885 - loss: 1.3867 - val_accuracy: 0.5787 - val_loss: 1.1411 - val_macro_f1: 0.3784 - learning_rate: 0.0010
Epoch 3/8
  val_macro_f1 0.4525
224/224 - 44s - 196ms/step - accuracy: 0.5332 - loss: 1.2105 - val_accuracy: 0.6466 - val_loss: 1.0030 - val_macro_f1: 0.4525 - learning_rate: 0.0010
Epoch 4/8
  val_macro_f1 0.4201
224/224 - 45s - 200ms/step - accuracy: 0.5555 - loss: 1.1794 - val_accuracy: 0.5304 - val_loss: 1.2182 - val_macro_f1: 0.4201 - learning_rate: 0.0010
Epoch 5/8
  val_macro_f1 0.4235
224/224 - 45s - 199ms/step - accuracy: 0.5542 - loss: 1.0958 - val_accuracy: 0.5934 - val_loss: 1.1280 - val_macro_f1: 0.4235 - learning_rate: 0.0010
Epoch 6/8
  val_macro_f1 0.4620
224/224 - 83s - 372ms/step - accuracy: 0.6091 - loss: 0.9839 - val_a

2026-08-03 09:30:58.095548: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-03 09:30:58.228507: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


-> macro-F1 0.5550 | acc 0.6960 | bal-acc 0.6764 | 27.4 min


### 6b. Leakage ablation

The same models retrained on the naive image-level split. The gap between the two rows is
your headline methodological result — and the reason published numbers on this dataset run
high.

In [10]:
for name in RUN_ABLATION_MODELS:
    if name not in ZOO or (name, "naive") in done:
        continue
    print(f"\n{'='*70}\n{name}  [NAIVE image-level split]\n{'='*70}")
    t = time.time()
    r, _, _ = run_model(name, "naive_split", "naive")
    r["minutes"] = round((time.time() - t) / 60, 1)
    pd.DataFrame([r]).to_csv(RES, mode="a", header=not RES.exists(), index=False)
    print(f"-> macro-F1 {r['test_macro_f1']:.4f} | acc {r['test_acc']:.4f}")


DenseNet121  [NAIVE image-level split]
Epoch 1/8
  val_macro_f1 0.3711
224/224 - 111s - 495ms/step - accuracy: 0.3705 - loss: 1.8215 - val_accuracy: 0.5374 - val_loss: 1.2811 - val_macro_f1: 0.3711 - learning_rate: 0.0010
Epoch 2/8
  val_macro_f1 0.3763
224/224 - 34s - 150ms/step - accuracy: 0.4736 - loss: 1.4259 - val_accuracy: 0.6004 - val_loss: 1.0532 - val_macro_f1: 0.3763 - learning_rate: 0.0010
Epoch 3/8
  val_macro_f1 0.3689
224/224 - 34s - 154ms/step - accuracy: 0.5009 - loss: 1.3610 - val_accuracy: 0.4409 - val_loss: 1.3990 - val_macro_f1: 0.3689 - learning_rate: 0.0010
Epoch 4/8
  val_macro_f1 0.4075
224/224 - 36s - 162ms/step - accuracy: 0.5026 - loss: 1.3039 - val_accuracy: 0.6074 - val_loss: 1.0640 - val_macro_f1: 0.4075 - learning_rate: 0.0010
Epoch 5/8
  val_macro_f1 0.3598
224/224 - 36s - 159ms/step - accuracy: 0.5271 - loss: 1.2359 - val_accuracy: 0.5199 - val_loss: 1.2293 - val_macro_f1: 0.3598 - learning_rate: 0.0010
Epoch 6/8
  val_macro_f1 0.4288
224/224 - 37s - 1

## 7. Results

In [11]:
df = pd.read_csv(RES).drop_duplicates(["model", "split"], keep="last")
les = df[df["split"] == "lesion"].sort_values("test_macro_f1", ascending=False)

print("LESION-AWARE SPLIT — ranked by macro-F1\n")
cols = ["model", "params_M", "test_acc", "test_macro_f1", "test_balanced_acc",
        "test_macro_auc", "minutes"]
print(les[cols].to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nmajority-class baseline accuracy: {dist.max()/len(meta):.4f}")

nai = df[df["split"] == "naive"]
if len(nai):
    print("\n\nLEAKAGE ABLATION\n")
    cmp = (les.set_index("model")[["test_acc", "test_macro_f1"]]
             .join(nai.set_index("model")[["test_acc", "test_macro_f1"]],
                   lsuffix="_lesion", rsuffix="_naive", how="inner"))
    cmp["acc_inflation"] = cmp["test_acc_naive"] - cmp["test_acc_lesion"]
    cmp["f1_inflation"]  = cmp["test_macro_f1_naive"] - cmp["test_macro_f1_lesion"]
    print(cmp.to_string(float_format=lambda v: f"{v:.4f}"))

best = les.iloc[0]["model"]
print(f"\nBEST BACKBONE: {best}  ->  carry this one into Stage 3 (XAI)")
json.dump({"best_model": best, "classes": CLASSES,
           "majority_baseline": float(dist.max()/len(meta))},
          open(OUT / "stage2_summary.json", "w"), indent=2)

LESION-AWARE SPLIT — ranked by macro-F1

            model  params_M  test_acc  test_macro_f1  test_balanced_acc  test_macro_auc  minutes
         ResNet50   24.1100    0.7280         0.5971             0.6270          0.9491  14.8000
      DenseNet201   18.8200    0.6981         0.5595             0.6313          0.9359  21.1000
     ConvNeXtTiny   28.0200    0.6960         0.5550             0.6764          0.9340  27.4000
   EfficientNetB0    4.3800    0.6870         0.5426             0.6400          0.9257  12.2000
            VGG16   14.8500    0.6884         0.5423             0.6669          0.9291  12.6000
InceptionResNetV2   54.7300    0.6877         0.5394             0.6142          0.9191  23.4000
      MobileNetV2    2.5900    0.6926         0.5349             0.6406          0.9177  13.2000
      DenseNet121    7.3000    0.6447         0.5015             0.6421          0.9224  14.6000
      InceptionV3   22.3300    0.6565         0.4979             0.6135          0.906

In [12]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
o = les.sort_values("test_macro_f1")
yp = np.arange(len(o))
ax[0].barh(yp - .2, o["test_macro_f1"], .4, label="macro-F1", color="#4C72B0")
ax[0].barh(yp + .2, o["test_acc"], .4, label="accuracy", color="#C8CFE0")
ax[0].axvline(dist.max()/len(meta), color="crimson", ls="--", label="majority baseline")
ax[0].set_yticks(yp); ax[0].set_yticklabels(o["model"]); ax[0].legend()
ax[0].set_xlabel("score"); ax[0].set_title("Backbone comparison (lesion-aware split)")

pc = les.set_index("model")[[f"f1_{c}" for c in CLASSES]]
im = ax[1].imshow(pc.values, cmap="viridis", aspect="auto", vmin=0, vmax=1)
ax[1].set_xticks(range(len(CLASSES))); ax[1].set_xticklabels(CLASSES, rotation=45)
ax[1].set_yticks(range(len(pc))); ax[1].set_yticklabels(pc.index, fontsize=8)
ax[1].set_title("Per-class F1"); plt.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.savefig(OUT / "figs" / "fig3_benchmark.png", dpi=200); plt.show()

if best in preds:
    p, yt = preds[best]
    print(classification_report(yt, p, target_names=CLASSES, digits=4, zero_division=0))
    cm = confusion_matrix(yt, p, normalize="true")
    plt.figure(figsize=(7, 6))
    plt.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    plt.xticks(range(len(CLASSES)), CLASSES, rotation=45)
    plt.yticks(range(len(CLASSES)), CLASSES)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            plt.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                     color="white" if cm[i, j] > .5 else "black", fontsize=8)
    plt.title(f"{best} — row-normalised confusion matrix"); plt.colorbar()
    plt.tight_layout(); plt.savefig(OUT / "figs" / "fig4_confusion.png", dpi=200); plt.show()

              precision    recall  f1-score   support

       akiec     0.3462    0.5294    0.4186        34
         bcc     0.6395    0.6180    0.6286        89
         bkl     0.5850    0.6964    0.6359       168
          df     0.4000    0.4444    0.4211         9
         mel     0.3114    0.6164    0.4138       146
          nv     0.9518    0.7700    0.8513       974
        vasc     0.9375    0.7143    0.8108        21

    accuracy                         0.7280      1441
   macro avg     0.5959    0.6270    0.5971      1441
weighted avg     0.8069    0.7280    0.7546      1441



## Reading the results

**Expect macro-F1 in the 0.60–0.75 band and accuracy in the 0.78–0.86 band.** Published
lesion-level corrections on HAM10000 land near 78% accuracy. If you see accuracy above 0.92
on the lesion-aware split, something is wrong — re-check the integrity gate first.

**`df` and `vasc` will be the weak classes** (115 and 142 images). Low F1 there is expected
and honest; do not tune it away by fishing for a seed that looks better.

**The ablation gap is the finding.** Naive-split accuracy should sit several points above
lesion-aware. That gap, measured on your own runs, is what justifies the whole protocol.

**Two caveats worth a sentence each in the paper.** Roughly 18 duplicate pairs in HAM10000
are known to be missed by `lesion_id` — a small residual your protocol does not catch, and
saying so signals you know the literature. And these are single-split results; if time
permits, re-run the top three backbones across three seeds and report mean ± std, which is
what turns a table into evidence.

Stage 3 takes the winning backbone into Grad-CAM, Grad-CAM++ and LIME with
deletion/insertion faithfulness metrics.